In [1]:
!nvidia-smi

Sat Aug 30 15:22:55 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 572.83                 Driver Version: 572.83         CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090      WDDM  |   00000000:01:00.0  On |                  Off |
|  0%   32C    P8             15W /  450W |    1150MiB /  24564MiB |      9%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [3]:
import wandb

# export WANDB_API_KEY=7855589e123f4caee4595a43d22c86cce82fbda1
wandb.login(key="7855589e123f4caee4595a43d22c86cce82fbda1",relogin=True)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\mza0288\_netrc
wandb: Currently logged in as: mmabappy1 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [7]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from ultralytics import YOLO
import torch
import os
import pandas as pd

models = ["yolov8n-seg.pt", "yolov8s-seg.pt", "yolov8m-seg.pt", "yolov8l-seg.pt", "yolo11n-seg.pt", "yolov11s-seg.pt", "yolov11m-seg.pt", "yolov11l-seg.pt"]
data_path = "C:/data/data_set/data.yaml"

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

epochs = 200
run_name = "None"

for model_name in models:
    print(f"\nTraining model: {model_name}")


    batch_size = 16
    imgsz = 640

    # Clear CUDA cache before starting new training (if using CUDA)
    if device == "cuda":
        torch.cuda.empty_cache()
    
    try:
        model = YOLO(model_name).to(device)
    except Exception as e:
        print(f"An error occurred: {e}")

    run_name = f"{model_name.split('.')[0]}-bs{batch_size}-ep{epochs}"

    results = model.train(
        data=data_path,
        epochs=epochs,
        batch=batch_size,
        imgsz=imgsz,
        device=device,
        seed=7,
        optimizer="SGD",
        project="runs/poultry",
        name=run_name,
        save=True,
        verbose=True,
    )

    # Initialize wandb before training (your existing init)
    wandb.init(project="poultry", name=run_name)
    
    
    # Path to training metrics CSV created by ultralytics:
    # Usually: runs/train/<run_name>/metrics.csv
    metrics_csv_path = f"runs/poultry/{run_name}/results.csv"

    # Load the CSV into DataFrame
    df = pd.read_csv(metrics_csv_path)
    
    # Iterate over each epoch's metrics row
    for _, row in df.iterrows():
        # Convert epoch to int for clarity
        epoch_num = int(row['epoch'])
    
        # Prepare dictionary of metrics, exclude epoch key
        metrics = {k: float(v) for k, v in row.items() if k != 'epoch'}
    
        # Add epoch number to the dictionary for logging
        metrics['epoch'] = epoch_num
    
        # Log metrics for this epoch to wandb
        wandb.log(metrics)
    
    wandb.finish()

Using device: cuda

Training model: yolov8n-seg.pt


100%|██████████| 6.74M/6.74M [00:00<00:00, 11.1MB/s]


New https://pypi.org/project/ultralytics/8.3.189 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.171  Python-3.12.9 torch-2.6.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4090, 24564MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:/data/data_set/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8n-seg-bs16-ep200, nbs=64, nms=Fal

train: Scanning C:\data\data_set\train\labels.cache... 4824 images, 0 backgrounds, 0 corrupt: 100%|██████████| 4824/4824 [00:00<?, ?it/s]Scanning C:\data\data_set\train\labels.cache... 4824 images, 0 backgrounds, 0 corrupt: 100%|██████████| 4824/4824 [00:00<?, ?it/s]


val: Fast image access  (ping: 0.10.0 ms, read: 233.047.0 MB/s, size: 1755.9 KB)


...

    200/200      3.55G     0.3914     0.5447     0.2686     0.7989         36        640: 100%|██████████| 302/302 [00:30<00:00, 10.01it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 22/22 [00:04<00:00,  4.96it/s]


                   all        702       3143      0.973      0.971      0.984      0.789      0.961      0.959      0.967      0.596

200 epochs completed in 1.967 hours.
Optimizer stripped from runs\poultry\yolov11l-seg-bs16-ep200\weights\last.pt, 6.0MB
Optimizer stripped from runs\poultry\yolov11l-seg-bs16-ep200\weights\best.pt, 6.0MB

Validating runs\poultry\yolov11l-seg-bs16-ep200\weights\best.pt...
Ultralytics 8.3.171  Python-3.12.9 torch-2.6.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4090, 24564MiB)
YOLO11n-seg summary (fused): 113 layers, 2,834,958 parameters, 0 gradients, 10.2 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 22/22 [00:05<00:00,  4.37it/s]


                   all        702       3143      0.972      0.969      0.983      0.795      0.963       0.96      0.973      0.601
       broiler breeder         68        193      0.958      0.964      0.977      0.706      0.943      0.948      0.958      0.493
         broiler house        645       2950      0.985      0.974       0.99      0.883      0.983      0.971      0.988      0.709
Speed: 0.1ms preprocess, 1.0ms inference, 0.0ms loss, 1.7ms postprocess per image
Results saved to runs\poultry\yolov11l-seg-bs16-ep200


epoch,▁▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▇▇▇▇▇▇█
lr/pg0,█████▇▇▇▇▇▇▇▇▆▆▆▆▆▆▆▅▅▅▅▅▄▄▄▄▃▃▃▃▂▂▂▂▂▂▁
lr/pg1,▆███▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁
lr/pg2,▃████▇▇▇▇▇▇▇▆▆▆▆▆▆▆▅▅▅▅▄▄▄▄▃▃▃▃▂▂▂▂▂▁▁▁▁
metrics/mAP50(B),▆▆▆▆▁▆▇▅▇▇▇▂▃▆▅▆▇▇█▇▆▄▆▅▇▄▅▅▇▆▆▆▇▇▇▇▇▇▇▇
metrics/mAP50(M),▆█▄▂▇▅▅▄▁▆▃▄▆▅▇▇▆▆▇▆▅▇▆▆▆▇▇▅▇▇▆▇▇▇▇▇▆▆▆▆
metrics/mAP50-95(B),▅▅▅▅▄▅▁▅▅▄▃▆▇▆▆▆▇▆▇▆█▇██▇▆██▇██▇██▇███▇▇
metrics/mAP50-95(M),▅▅▃▂▆▁▃▁▅▂▄▄▂▄▄▅▅▃▅▄▆▆▇▇▇▇▆▇▇▇▇█████▇▇▇▇
metrics/precision(B),▄▄▇▁▂▃▆▃▃▃▄▁▇█▂▅▅▃▇▅▄▇▃▅▂▄▅▆▆▂▃▄▅▁▅▆▅▅▄▄
metrics/precision(M),▆▄▂▅█▄▅▃▆▇▁▅▅▄▄▃▆▄▄▆▇▅▅▆▅▇▅▅▆▆▆▆▃▅▄▅▅▅▅▅
metrics/recall(B),▅▆▆▃▁▆▅▅▁▄▆▅▆▄▆▆█▅▅▅██▇▆▆▅▄▅▆▆▆▆██▇▆▆▆▆▇
